In [1]:
pip install tk


Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for tk from https://files.pythonhosted.org/packages/1e/0b/029cbdb868bb555fed99bf6540fff072d500b3f895873709f25084e85e33/tk-0.1.0-py3-none-any.whl.metadata
Note: you may need to restart the kernel to use updated packages.


In [1]:
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import MinMaxScaler


# Function to preprocess data
def preprocess_data(data):
    label_encoder = LabelEncoder()
    data['gender'] = label_encoder.fit_transform(data['gender'])
    data['ever_married'] = label_encoder.fit_transform(data['ever_married'])
    data['work_type'] = label_encoder.fit_transform(data['work_type'])
    data['Residence_type'] = label_encoder.fit_transform(data['Residence_type'])
    data['smoking_status'] = label_encoder.fit_transform(data['smoking_status'])

    # Filling null values with mean
    data = data.fillna(data.mean())

    return data


# Function to balance dataset using SMOTE
def balance_dataset(X, y):
    sm = SMOTE(random_state=42)
    X_res, y_res = sm.fit_resample(X, y)
    return X_res, y_res


# Function to train logistic regression model
def train_logistic_regression(X_train, y_train):
    lr = LogisticRegression(max_iter=400)
    lr.fit(X_train, y_train)
    return lr


# Function to train random forest model
def train_random_forest(X_train, y_train):
    classifier_rf = RandomForestClassifier(n_estimators=100, random_state=42)
    classifier_rf.fit(X_train, y_train)
    return classifier_rf


# Function to preprocess input and make predictions
def make_predictions(model, input_data):
    input_data = preprocess_data(input_data)
    X = input_data.drop(['stroke', 'id'], axis=1)
    X_res, _ = balance_dataset(X, [])
    scaler = MinMaxScaler()
    X_res = scaler.fit_transform(X_res)
    return model.predict(X_res)


# Function to handle button click event
def on_predict():
    # Get input values from entry widgets
    gender = gender_var.get()
    age = age_var.get()
    hypertension = hypertension_var.get()
    heart_disease = heart_disease_var.get()
    ever_married = ever_married_var.get()
    work_type = work_type_var.get()
    Residence_type = Residence_type_var.get()
    avg_glucose_level = avg_glucose_level_var.get()
    bmi = bmi_var.get()
    smoking_status = smoking_status_var.get()

    # Create a dictionary with input values
    input_data = {
        'gender': [gender],
        'age': [float(age)],
        'hypertension': [hypertension],
        'heart_disease': [heart_disease],
        'ever_married': [ever_married],
        'work_type': [work_type],
        'Residence_type': [Residence_type],
        'avg_glucose_level': [float(avg_glucose_level)],
        'bmi': [float(bmi)],
        'smoking_status': [smoking_status]
    }

    # Create a DataFrame from input data
    input_df = pd.DataFrame(input_data)

    # Make predictions using selected model
    if model_choice.get() == 'Logistic Regression':
        predictions = make_predictions(logistic_regression_model, input_df)
    elif model_choice.get() == 'Random Forest':
        predictions = make_predictions(random_forest_model, input_df)

    # Show prediction result
    messagebox.showinfo("Prediction Result", f"Stroke Prediction: {predictions[0]}")


# Load the dataset
data = pd.read_csv('healthcare-dataset-stroke-data.csv')

# Preprocess data
data = preprocess_data(data)

# Split data into features and target
X = data.drop(['stroke', 'id'], axis=1)
y = data['stroke']

# Balance dataset
X_res, y_res = balance_dataset(X, y)

# Train Logistic Regression model
logistic_regression_model = train_logistic_regression(X_res, y_res)

# Train Random Forest model
random_forest_model = train_random_forest(X_res, y_res)

# Create GUI window
window = tk.Tk()
window.title("Stroke Prediction")

# Create labels and entry widgets for input features
tk.Label(window, text="Gender (Male=0, Female=1)").grid(row=0, column=0)
gender_var = tk.StringVar()
tk.Entry(window, textvariable=gender_var).grid(row=0, column=1)

# Add more labels and entry widgets for other input features...
tk.Label(window, text="Age").grid(row=1, column=0)
age_var = tk.StringVar()
tk.Entry(window, textvariable=age_var).grid(row=1, column=1)

tk.Label(window, text="Hypertension (0=No, 1=Yes)").grid(row=2, column=0)
hypertension_var = tk.StringVar()
tk.Entry(window, textvariable=hypertension_var).grid(row=2, column=1)

tk.Label(window, text="Heart Disease (0=No, 1=Yes)").grid(row=3, column=0)
heart_disease_var = tk.StringVar()
tk.Entry(window, textvariable=heart_disease_var).grid(row=3, column=1)

tk.Label(window, text="Ever Married (0=No, 1=Yes)").grid(row=4, column=0)
ever_married_var = tk.StringVar()
tk.Entry(window, textvariable=ever_married_var).grid(row=4, column=1)

# Dropdown menu for Work Type
tk.Label(window, text="Work Type").grid(row=5, column=0)
work_type_var = tk.StringVar()
work_type_choices = ['Private', 'Self-employed', 'Govt_job', 'Children', 'Never_worked']
work_type_menu = ttk.OptionMenu(window, work_type_var, *work_type_choices)
work_type_menu.grid(row=5, column=1)

tk.Label(window, text="Residence Type (0=Urban, 1=Rural)").grid(row=6, column=0)
Residence_type_var = tk.StringVar()
tk.Entry(window, textvariable=Residence_type_var).grid(row=6, column=1)

tk.Label(window, text="Average Glucose Level").grid(row=7, column=0)
avg_glucose_level_var = tk.StringVar()
tk.Entry(window, textvariable=avg_glucose_level_var).grid(row=7, column=1)

tk.Label(window, text="BMI").grid(row=8, column=0)
bmi_var = tk.StringVar()
tk.Entry(window, textvariable=bmi_var).grid(row=8, column=1)

tk.Label(window, text="Smoking Status (0=Unknown, 1=Never smoked, 2=formerly smoked, 3=smokes)").grid(row=9, column=0)
smoking_status_var = tk.StringVar()
tk.Entry(window, textvariable=smoking_status_var).grid(row=9, column=1)

# Create drop-down menu for selecting model
tk.Label(window, text="Select Model:").grid(row=10, column=0)
model_choice = ttk.Combobox(window, values=['Logistic Regression', 'Random Forest'])
model_choice.grid(row=10, column=1)
model_choice.current(0)

# Button to predict
predict_button = tk.Button(window, text="Predict", command=on_predict)
predict_button.grid(row=11, column=1)

window.mainloop()


Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\tkinter\__init__.py", line 1948, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\Temp\ipykernel_6648\1880594513.py", line 95, in on_predict
    predictions = make_predictions(random_forest_model, input_df)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\Temp\ipykernel_6648\1880594513.py", line 53, in make_predictions
    X = input_data.drop(['stroke', 'id'], axis=1)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\site-packages\pandas\core\frame.py", line 5258, in drop
    return super().drop(
           ^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\site-packages\pandas\core\generic.py", line 4549, in drop
    obj = obj._drop_axis(labels, axis, level=level, errors=errors)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [3]:
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import MinMaxScaler


# Function to preprocess data
def preprocess_data(data):
    label_encoder = LabelEncoder()
    data['gender'] = label_encoder.fit_transform(data['gender'])
    data['ever_married'] = label_encoder.fit_transform(data['ever_married'])
    data['work_type'] = label_encoder.fit_transform(data['work_type'])
    data['Residence_type'] = label_encoder.fit_transform(data['Residence_type'])
    data['smoking_status'] = label_encoder.fit_transform(data['smoking_status'])

    # Filling null values with mean
    data = data.fillna(data.mean())

    return data


# Function to balance dataset using SMOTE
def balance_dataset(X, y):
    sm = SMOTE(random_state=42)
    X_res, y_res = sm.fit_resample(X, y)
    return X_res, y_res


# Function to train logistic regression model
def train_logistic_regression(X_train, y_train):
    lr = LogisticRegression(max_iter=400)
    lr.fit(X_train, y_train)
    return lr


# Function to train random forest model
def train_random_forest(X_train, y_train):
    classifier_rf = RandomForestClassifier(n_estimators=100, random_state=42)
    classifier_rf.fit(X_train, y_train)
    return classifier_rf


# Function to preprocess input and make predictions
def make_predictions(model, input_data):
    input_data = preprocess_data(input_data)
    X = input_data.drop(['id'], axis=1)  # Drop 'id' column
    X_res, _ = balance_dataset(X, [])  # Balance dataset
    scaler = MinMaxScaler()
    X_res = scaler.fit_transform(X_res)
    return model.predict(X_res)


# Function to handle button click event
def on_predict():
    # Get input values from entry widgets
    gender = gender_var.get()
    age = age_var.get()
    hypertension = hypertension_var.get()
    heart_disease = heart_disease_var.get()
    ever_married = ever_married_var.get()
    work_type = work_type_var.get()
    Residence_type = Residence_type_var.get()
    avg_glucose_level = avg_glucose_level_var.get()
    bmi = bmi_var.get()
    smoking_status = smoking_status_var.get()

    # Check if any input field is empty
    if '' in [gender, age, hypertension, heart_disease, ever_married, work_type,
               Residence_type, avg_glucose_level, bmi, smoking_status]:
        messagebox.showerror("Error", "Please fill in all input fields.")
        return

    # Create a dictionary with input values
    input_data = {
        'gender': [gender],
        'age': [float(age)],
        'hypertension': [hypertension],
        'heart_disease': [heart_disease],
        'ever_married': [ever_married],
        'work_type': [work_type],
        'Residence_type': [Residence_type],
        'avg_glucose_level': [float(avg_glucose_level)],
        'bmi': [float(bmi)],
        'smoking_status': [smoking_status]
    }

    # Create a DataFrame from input data
    input_df = pd.DataFrame(input_data)

    # Make predictions using selected model
    if model_choice.get() == 'Logistic Regression':
        predictions = make_predictions(logistic_regression_model, input_df)
    elif model_choice.get() == 'Random Forest':
        predictions = make_predictions(random_forest_model, input_df)

    # Show prediction result
    messagebox.showinfo("Prediction Result", f"Stroke Prediction: {predictions[0]}")


# Load the dataset
data = pd.read_csv('healthcare-dataset-stroke-data.csv')

# Preprocess data
data = preprocess_data(data)

# Split data into features and target
X = data.drop(['stroke'], axis=1)
y = data['stroke']

# Balance dataset
X_res, y_res = balance_dataset(X, y)

# Train Logistic Regression model
logistic_regression_model = train_logistic_regression(X_res, y_res)

# Train Random Forest model
random_forest_model = train_random_forest(X_res, y_res)

# Create GUI window
window = tk.Tk()
window.title("Stroke Prediction")

# Create labels and entry widgets for input features
tk.Label(window, text="Gender (Male=0, Female=1)").grid(row=0, column=0)
gender_var = tk.StringVar()
tk.Entry(window, textvariable=gender_var).grid(row=0, column=1)

# Add more labels and entry widgets for other input features...
tk.Label(window, text="Age").grid(row=1, column=0)
age_var = tk.StringVar()
tk.Entry(window, textvariable=age_var).grid(row=1, column=1)

tk.Label(window, text="Hypertension (0=No, 1=Yes)").grid(row=2, column=0)
hypertension_var = tk.StringVar()
tk.Entry(window, textvariable=hypertension_var).grid(row=2, column=1)

tk.Label(window, text="Heart Disease (0=No, 1=Yes)").grid(row=3, column=0)
heart_disease_var = tk.StringVar()
tk.Entry(window, textvariable=heart_disease_var).grid(row=3, column=1)

tk.Label(window, text="Ever Married (0=No, 1=Yes)").grid(row=4, column=0)
ever_married_var = tk.StringVar()
tk.Entry(window, textvariable=ever_married_var).grid(row=4, column=1)

# Dropdown menu for Work Type
tk.Label(window, text="Work Type").grid(row=5, column=0)
work_type_var = tk.StringVar()
work_type_choices = ['Private', 'Self-employed', 'Govt_job', 'Children', 'Never_worked']
work_type_menu = ttk.OptionMenu(window, work_type_var, *work_type_choices)
work_type_menu.grid(row=5, column=1)

tk.Label(window, text="Residence Type (0=Urban, 1=Rural)").grid(row=6, column=0)
Residence_type_var = tk.StringVar()
tk.Entry(window, textvariable=Residence_type_var).grid(row=6, column=1)

tk.Label(window, text="Average Glucose Level").grid(row=7, column=0)
avg_glucose_level_var = tk.StringVar()
tk.Entry(window, textvariable=avg_glucose_level_var).grid(row=7, column=1)

tk.Label(window, text="BMI").grid(row=8, column=0)
bmi_var = tk.StringVar()
tk.Entry(window, textvariable=bmi_var).grid(row=8, column=1)

tk.Label(window, text="Smoking Status (0=Unknown, 1=Never smoked, 2=formerly smoked, 3=smokes)").grid(row=9, column=0)
smoking_status_var = tk.StringVar()
tk.Entry(window, textvariable=smoking_status_var).grid(row=9, column=1)

# Create drop-down menu for selecting model
tk.Label(window, text="Select Model:").grid(row=10, column=0)
model_choice = ttk.Combobox(window, values=['Logistic Regression', 'Random Forest'])
model_choice.grid(row=10, column=1)
model_choice.current(0)

# Button to predict
predict_button = tk.Button(window, text="Predict", command=on_predict)
predict_button.grid(row=11, column=1)

window.mainloop()


C:\Users\DELL\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\tkinter\__init__.py", line 1948, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\Temp\ipykernel_6648\2327292039.py", line 101, in on_predict
    predictions = make_predictions(random_forest_model, input_df)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\T

In [7]:
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from imblearn.over_sampling import SMOTE

# Function to preprocess data
def preprocess_data(data):
    label_encoder = LabelEncoder()
    data['gender'] = label_encoder.fit_transform(data['gender'])
    data['ever_married'] = label_encoder.fit_transform(data['ever_married'])
    data['work_type'] = label_encoder.fit_transform(data['work_type'])
    data['Residence_type'] = label_encoder.fit_transform(data['Residence_type'])
    data['smoking_status'] = label_encoder.fit_transform(data['smoking_status'])

    # Filling null values with mean
    data = data.fillna(data.mean())

    return data

# Function to balance dataset using SMOTE
def balance_dataset(X, y):
    sm = SMOTE(random_state=42)
    X_res, y_res = sm.fit_resample(X, y)
    return X_res, y_res

# Function to train logistic regression model
def train_logistic_regression(X_train, y_train):
    lr = LogisticRegression(max_iter=1000)  # Increase max_iter
    lr.fit(X_train, y_train)
    return lr

# Function to train random forest model
def train_random_forest(X_train, y_train):
    classifier_rf = RandomForestClassifier(n_estimators=100, random_state=42)
    classifier_rf.fit(X_train, y_train)
    return classifier_rf

# Function to preprocess input and make predictions
def make_predictions(model, input_data):
    input_data = preprocess_data(input_data)
    scaler = MinMaxScaler()
    X_res = scaler.fit_transform(input_data)
    return model.predict(X_res)

# Function to handle button click event
def on_predict():
    # Get input values from entry widgets
    gender = gender_var.get()
    age = age_var.get()
    hypertension = hypertension_var.get()
    heart_disease = heart_disease_var.get()
    ever_married = ever_married_var.get()
    work_type = work_type_var.get()
    Residence_type = Residence_type_var.get()
    avg_glucose_level = avg_glucose_level_var.get()
    bmi = bmi_var.get()
    smoking_status = smoking_status_var.get()

    # Check if any input field is empty
    if '' in [gender, age, hypertension, heart_disease, ever_married, work_type,
               Residence_type, avg_glucose_level, bmi, smoking_status]:
        messagebox.showerror("Error", "Please fill in all input fields.")
        return

    # Create a dictionary with input values
    input_data = {
        'gender': [gender],
        'age': [float(age)],
        'hypertension': [hypertension],
        'heart_disease': [heart_disease],
        'ever_married': [ever_married],
        'work_type': [work_type],
        'Residence_type': [Residence_type],
        'avg_glucose_level': [float(avg_glucose_level)],
        'bmi': [float(bmi)],
        'smoking_status': [smoking_status]
    }

    # Create a DataFrame from input data
    input_df = pd.DataFrame(input_data)

    # Make predictions using selected model
    if model_choice.get() == 'Logistic Regression':
        predictions = make_predictions(logistic_regression_model, input_df)
    elif model_choice.get() == 'Random Forest':
        predictions = make_predictions(random_forest_model, input_df)

    # Show prediction result
    messagebox.showinfo("Prediction Result", f"Stroke Prediction: {predictions[0]}")

# Load the dataset
data = pd.read_csv('healthcare-dataset-stroke-data.csv')

# Preprocess data
data = preprocess_data(data)

# Split data into features and target
X = data.drop(['stroke'], axis=1)
y = data['stroke']

# Balance dataset
X_res, y_res = balance_dataset(X, y)

# Train Logistic Regression model
logistic_regression_model = train_logistic_regression(X_res, y_res)

# Train Random Forest model
random_forest_model = train_random_forest(X_res, y_res)

# Create GUI window
window = tk.Tk()
window.title("Stroke Prediction")

# Create labels and entry widgets for input features
tk.Label(window, text="Gender (Male=0, Female=1)").grid(row=0, column=0)
gender_var = tk.StringVar()
tk.Entry(window, textvariable=gender_var).grid(row=0, column=1)

# Add more labels and entry widgets for other input features...
tk.Label(window, text="Age").grid(row=1, column=0)
age_var = tk.StringVar()
tk.Entry(window, textvariable=age_var).grid(row=1, column=1)

tk.Label(window, text="Hypertension (0=No, 1=Yes)").grid(row=2, column=0)
hypertension_var = tk.StringVar()
tk.Entry(window, textvariable=hypertension_var).grid(row=2, column=1)

tk.Label(window, text="Heart Disease (0=No, 1=Yes)").grid(row=3, column=0)
heart_disease_var = tk.StringVar()
tk.Entry(window, textvariable=heart_disease_var).grid(row=3, column=1)

tk.Label(window, text="Ever Married (0=No, 1=Yes)").grid(row=4, column=0)
ever_married_var = tk.StringVar()
tk.Entry(window, textvariable=ever_married_var).grid(row=4, column=1)

# Dropdown menu for Work Type
tk.Label(window, text="Work Type").grid(row=5, column=0)
work_type_var = tk.StringVar()
work_type_choices = ['Private', 'Self-employed', 'Govt_job', 'children', 'Never_worked']
work_type_menu = ttk.OptionMenu(window, work_type_var, *work_type_choices)
work_type_menu.grid(row=5, column=1)

tk.Label(window, text="Residence Type (0=Urban, 1=Rural)").grid(row=6, column=0)
Residence_type_var = tk.StringVar()
tk.Entry(window, textvariable=Residence_type_var).grid(row=6, column=1)

tk.Label(window, text="Average Glucose Level").grid(row=7, column=0)
avg_glucose_level_var = tk.StringVar()
tk.Entry(window, textvariable=avg_glucose_level_var).grid(row=7, column=1)

tk.Label(window, text="BMI").grid(row=8, column=0)
bmi_var = tk.StringVar()
tk.Entry(window, textvariable=bmi_var).grid(row=8, column=1)

tk.Label(window, text="Smoking Status (0=Unknown, 1=Never smoked, 2=Formerly smoked, 3=Smokes)").grid(row=9, column=0)
smoking_status_var = tk.StringVar()
tk.Entry(window, textvariable=smoking_status_var).grid(row=9, column=1)

# Create drop-down menu for selecting model
tk.Label(window, text="Select Model:").grid(row=10, column=0)
model_choice = ttk.Combobox(window, values=['Logistic Regression', 'Random Forest'])
model_choice.grid(row=10, column=1)
model_choice.current(0)

# Button to predict
predict_button = tk.Button(window, text="Predict", command=on_predict)
predict_button.grid(row=11, column=1)

window.mainloop()


C:\Users\DELL\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\DELL\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\tkinter\__init__.py", line 1948, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\Temp\ipykernel_6648\268

In [8]:
pip install flask


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
from flask import Flask, render_template, request
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from imblearn.over_sampling import SMOTE
import warnings

app = Flask(__name__)

# Ignore the warning
warnings.filterwarnings("ignore", message="X does not have valid feature names, but MinMaxScaler was fitted with feature names")

# Load the dataset
data = pd.read_csv('healthcare-dataset-stroke-data.csv')

# Preprocess data
label_encoder = LabelEncoder()
data['gender'] = label_encoder.fit_transform(data['gender'])
data['ever_married'] = label_encoder.fit_transform(data['ever_married'])
data['work_type'] = label_encoder.fit_transform(data['work_type'])
data['Residence_type'] = label_encoder.fit_transform(data['Residence_type'])
data['smoking_status'] = label_encoder.fit_transform(data['smoking_status'])
data = data.fillna(data.mean())

# Split data into features and target
X = data.drop(['stroke', 'id'], axis=1)
y = data['stroke']

# Balance dataset
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)

# Train Logistic Regression model
lr = LogisticRegression(max_iter=400)
lr.fit(X_res, y_res)

# Train Random Forest model
classifier_rf = RandomForestClassifier(n_estimators=100, random_state=42)
classifier_rf.fit(X_res, y_res)

# Normalize the data
scaler = MinMaxScaler()
X_res = scaler.fit_transform(X_res)
X_res = scaler.transform(X_res)

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    if request.method == 'POST':
        # Retrieve user input from the form
        gender = request.form['gender']
        age = float(request.form['age'])
        hypertension = int(request.form['hypertension'])
        heart_disease = int(request.form['heart_disease'])
        ever_married = int(request.form['ever_married'])
        work_type = int(request.form['work_type'])
        Residence_type = int(request.form['Residence_type'])
        avg_glucose_level = float(request.form['avg_glucose_level'])
        bmi = float(request.form['bmi'])
        smoking_status = int(request.form['smoking_status'])

        # Preprocess input data
        input_data = preprocess_input(gender, age, hypertension, heart_disease,
                                      ever_married, work_type, Residence_type,
                                      avg_glucose_level, bmi, smoking_status)

        # Make predictions using both models
        prediction_lr = lr.predict(input_data)[0]
        prediction_rf = classifier_rf.predict(input_data)[0]

        return render_template('result.html', prediction_lr=prediction_lr, prediction_rf=prediction_rf)

def preprocess_input(gender, age, hypertension, heart_disease, ever_married,
                     work_type, Residence_type, avg_glucose_level, bmi, smoking_status):
    # Convert categorical variables to numerical using label encoding
    gender_encoded = label_encoder.transform([gender])
    # Encode other categorical variables...
    
    # Create a DataFrame from preprocessed input data
    input_df = pd.DataFrame({
        'gender': gender_encoded,
        'age': [age],
        'hypertension': [hypertension],
        'heart_disease': [heart_disease],
        'ever_married': [ever_married],
        'work_type': [work_type],
        'Residence_type': [Residence_type],
        'avg_glucose_level': [avg_glucose_level],
        'bmi': [bmi],
        'smoking_status': [smoking_status]
    })

    # Scale the input data
    input_scaled = scaler.transform(input_df)

    return input_scaled

if __name__ == '__main__':
    app.run(debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [11]:
!pip install flask_wtf

Defaulting to user installation because normal site-packages is not writeable


In [10]:
from flask import Flask, render_template, request
from flask_wtf import flaskForm
from wtforms import Stringfield, SubmitField

ImportError: cannot import name 'flaskForm' from 'flask_wtf' (C:\Users\DELL\AppData\Roaming\Python\Python311\site-packages\flask_wtf\__init__.py)